# Эмбеддинги и кластеризация BERTopic. Другие baseline-решения

In [ ]:
import sqlite3
import json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from gensim.models.coherencemodel import CoherenceModel
from gensim import corpora

## Загрузка данных из SQLite

In [25]:
db_path = Path.cwd().parent / 'data' / 'processed' / 'comments.db'
conn = sqlite3.connect(db_path)
df = pd.read_sql(
    "SELECT * FROM comments WHERE text IS NOT NULL AND TRIM(text) != ''",
    conn
)
conn.close()

print(df['product'].value_counts())

product
Видео-платформа    608
Маркетплейс        285
Name: count, dtype: int64


В рамках демонстрации работы модели будут использоваться комментарии сервисов Видео-платформа (август-ноябрь) и Маркетплейс (июнь-октябрь). Важно уточнить, что пайплайн при этом воспроизводим для всех продуктов.

In [26]:
PRODUCT = 'Маркетплейс'   # 'Видео-платформа'
df = df[df['product'] == PRODUCT].reset_index(drop=True)
print(f"Комментариев ({PRODUCT}): {len(df)}")
df.head(3)

Комментариев (Маркетплейс): 285


,id,date,product,platform,text
0,245,2025-10-01,Маркетплейс,WEB,Отсутствие поддержки. В МТС невозможно ни с ке...
1,246,2025-10-13,Маркетплейс,WEB,"Обман со времён работы магазина, что приводит ..."
2,247,2025-10-29,Маркетплейс,WEB,Ни когда раньше не пользовался


## Выбор модели эмбеддингов

`multilingual-e5-large`выбрана потому, что дает хорошее качество на русских текстах, и она спокойно развернется на 8 GB VRAM используемой видеокарты.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM свободно: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

Устройство: cuda
GPU: NVIDIA GeForce RTX 3060 Ti
VRAM свободно: 3.7 GB


In [28]:
MODEL_NAME = "intfloat/multilingual-e5-large"

model = SentenceTransformer(MODEL_NAME, device=device)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5181.74it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Генерация эмбеддингов

In [29]:
texts = df['text'].tolist()
texts_prefixed = ["passage: " + t for t in texts]

embeddings = model.encode(
    texts_prefixed,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    device=device
)

print(f"Матрица эмбеддингов: {embeddings.shape}")

Batches: 100%|██████████| 5/5 [00:07<00:00,  1.50s/it]

Матрица эмбеддингов: (285, 1024)


## Первый запуск BERTopic

In [ ]:
MIN_CLUSTER_SIZE = 7  # Видео-платформа=15, Маркетплейс=7
MIN_SAMPLES = 3       # Видео-платформа=7, Маркетплейс=4
N_COMPONENTS_UMAP = 5

In [47]:
umap_model = UMAP(
    n_components=N_COMPONENTS_UMAP,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

In [ ]:
russian_stopwords = [
    'и', 'в', 'не', 'что', 'на', 'я', 'с', 'как', 'а', 'то', 'все',
    'так', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по',
    'только', 'было', 'вот', 'от', 'еще', 'нет', 'о', 'из', 'ну', 'ли',
    'если', 'уже', 'или', 'ни', 'до', 'там', 'потом', 'себя', 'они',
    'тут', 'где', 'есть', 'для', 'мы', 'их', 'чем', 'без', 'раз', 'при',
    'про', 'это', 'со', 'им', 'его', 'ее', 'этот', 'того', 'этого',
]

vectorizer = CountVectorizer(
    stop_words=russian_stopwords,
    min_df=2,
    ngram_range=(1, 2)
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    language='multilingual',
    calculate_probabilities=True,
    verbose=True
)

In [49]:
topics, probs = topic_model.fit_transform(texts, embeddings)

n_topics = len(set(topics)) - 1
n_noise  = (np.array(topics) == -1).sum()
print()
print(f"Всего найдено тем: {n_topics}")
print(f"Шумовых документов (topic=-1): {n_noise} ({n_noise/len(topics)*100:.4f}%)")

2026-03-29 15:54:26,264 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-29 15:54:26,693 - BERTopic - Dimensionality - Completed ✓
2026-03-29 15:54:26,694 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 15:54:26,704 - BERTopic - Cluster - Completed ✓
2026-03-29 15:54:26,705 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 15:54:26,718 - BERTopic - Representation - Completed ✓



Всего найдено тем: 7
Шумовых документов (topic=-1): 29 (10.1754%)


In [50]:
topic_info = topic_model.get_topic_info()

print(f"Самые крупные темы:")
topic_info[topic_info['Topic'] != -1]

Самые крупные темы:


,Topic,Count,Name,Representation,Representative_Docs
1,0,142,0_доставки_самовывоз_наличии_товара,"[доставки, самовывоз, наличии, товара, товаров...",[Платный самовывоз. За вход в магазин еще введ...
2,1,35,1_товар_сразу_продаже_через,"[товар, сразу, продаже, через, зачем, новый, т...","[Нужные товар отсутствуют, нет возможности офо..."
3,2,28,2_заказ_товар_сказали_деньги,"[заказ, товар, сказали, деньги, итоге, день, м...","[Всё плохо, оформляешь доставку, товар не прив..."
4,3,16,3_цены_выше_очень_даже,"[цены, выше, очень, даже, доставка, продаете, ...","[ребят, у вас очень высокие цены, даже с учето..."
5,4,15,4_мтс_заказа_мне_магазин,"[мтс, заказа, мне, магазин, телефон, магазине,...",[Нет предлагаемых товаров в магазинах. На весь...
6,5,11,5_рассрочка_написано_месяца_рассрочку,"[рассрочка, написано, месяца, рассрочку, факту...",[Везде пишется про рассрочку 24 месяца но по ф...
7,6,9,6_новый_почему_всегда_телефона,"[новый, почему, всегда, телефона, купить, теле...",[Потому что нехорошо заниматься мошенничеством...


In [51]:
print("Ключевые слова тем:")
for _, row in topic_info[topic_info['Topic'] != -1].iterrows():
    tid = row['Topic']
    keywords = [w for w, _ in topic_model.get_topic(tid)]
    print(f"Тема {tid} ({int(row['Count'])} текстов): {', '.join(keywords[:8])}")

Ключевые слова тем:
Тема 0 (142 текстов): доставки, самовывоз, наличии, товара, товаров, платный самовывоз, платный, дорого
Тема 1 (35 текстов): товар, сразу, продаже, через, зачем, новый, товара, товары
Тема 2 (28 текстов): заказ, товар, сказали, деньги, итоге, день, магазина, чтобы
Тема 3 (16 текстов): цены, выше, очень, даже, доставка, продаете, мтс, платная
Тема 4 (15 текстов): мтс, заказа, мне, магазин, телефон, магазине, просто, ещё
Тема 5 (11 текстов): рассрочка, написано, месяца, рассрочку, факту, оформлении, можно, товар
Тема 6 (9 текстов): новый, почему, всегда, телефона, купить, телефон, система, данные


## Наблюдения

### Видео-платформа

- При запуске BERTopic на эмбеддингах, полученных из обезличенных данных, среди ключевых слов нередко появлялись междометия и союзы -> было решено исключить стоп-слова, что должно было сделать больше вклад более содержательных слов. После фильтрации текстов на стоп-слова среди ключевых слов действительно стало больше содержательного текста.
- MIN_CLUSTER_SIZE, равный 15, дал всего 7 тем. 4 из них были связаны общим мотивом: жалоба на ассортимент, интерфейс, рекламу и отмену подписки. Три темы получились довольно общими, причем самая большая объединила 250 слов.
- MIN_CLUSTER_SIZE, равный 10, дал уже больше тем, но некоторые из них оказались слишком размытыми.
- MIN_CLUSTER_SIZE от 10 до 15 не выводят на "золотую середину". По итогу более предпочтительным выглядит MIN_CLUSTER_SIZE=10 — он дает достаточную детализацию, которая будет полезна продуктовому менеджеру.
- MIN_SAMPLES=8 применялся для всех испытаний.

### Маркетплейс

- MIN_CLUSTER_SIZE, равный 10, выделил всего две темы, что объяснимо — 10 документов составляет ~3%, что уменьшает количество плотных областей для HDBSCAN и, как следствие, комментарии объединяются в два больших кластера.
- Были выбраны параметры MIN_CLUSTER_SIZE=5 и MIN_SAMPLES=3, соответствующие меньшему объему комментариев для продукта.

### Выводы
- Очевидна необходимость динамично определять параметры кластеризации. Зафиксировать их для всех продуктов будет некорректным, поскольку BERTopic либо не выделит темы продуктов с малым числом комментариев, либо раздробит комментарии на множество мелких подтем.

---
## Baseline-методы и метрики качества

Сравниваем BERTopic с классическими методами:
- **LDA** (Latent Dirichlet Allocation) — вероятностная тематическая модель на TF-IDF
- **NMF** (Non-negative Matrix Factorization) — матричная факторизация на TF-IDF
- **KMeans** — кластеризация на эмбеддингах

Метрики:
- **Silhouette score** — насколько плотны и разделены кластеры (BERTopic, KMeans)
- **Coherence (c_v)** — насколько слова одной темы семантически близки (BERTopic, LDA, NMF)

### Адаптивные параметры BERTopic

In [52]:
def get_bertopic_params(n_docs: int):
    min_cluster_size = max(5, n_docs // 40)
    min_samples = max(3, min_cluster_size // 2)
    return {'min_cluster_size': min_cluster_size, 'min_samples': min_samples}

params = get_bertopic_params(len(df))
print(f'Продукт: {PRODUCT}  ({len(df)} doc)')
print(f'min_cluster_size = {params["min_cluster_size"]}')
print(f'min_samples = {params["min_samples"]}')

Продукт: Маркетплейс  (285 doc)
min_cluster_size = 7
min_samples = 3


### Baseline: LDA

In [54]:
lda_vectorizer = CountVectorizer(
    stop_words=russian_stopwords, min_df=2, max_df=0.95
)
X_count = lda_vectorizer.fit_transform(texts)

# Для равноценного сравнения возьмем то же число тем, что нашёл BERTopic
lda = LatentDirichletAllocation(
    n_components=n_topics, random_state=42, max_iter=30, n_jobs=-1
)
lda_topic_matrix = lda.fit_transform(X_count)
lda_labels = lda_topic_matrix.argmax(axis=1)

In [55]:
lda_feature_names = lda_vectorizer.get_feature_names_out()
print('Ключевые слова (LDA):')
for tid, topic in enumerate(lda.components_):
    top_words = [lda_feature_names[i] for i in topic.argsort()[:-9:-1]]
    count = (lda_labels == tid).sum()
    print(f'Тема {tid} ({count} текстов): {", ".join(top_words)}')

Ключевые слова (LDA):
Тема 0 (54 текстов): товар, заказ, магазин, телефон, заказа, магазине, через, больше
Тема 1 (41 текстов): заказ, товар, наличии, телефон, деньги, товара, магазине, мне
Тема 2 (28 текстов): товара, могу, почему, срок, цены, заказа, наличии, телефон
Тема 3 (35 текстов): заказ, товар, сервис, можно, рассрочку, невозможно, сайт, очень
Тема 4 (40 текстов): мтс, товара, магазин, магазине, товар, выше, даже, цены
Тема 5 (39 текстов): товаров, самовывоз, просто, доставки, ещё, платный, магазин, мтс
Тема 6 (48 текстов): доставка, мтс, платная, плохо, всё, цены, магазинах, товара


### Baseline: NMF

In [56]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words=russian_stopwords, min_df=2, max_df=0.95
)
X_tfidf = tfidf_vectorizer.fit_transform(texts)

nmf = NMF(n_components=n_topics, random_state=42, max_iter=500)
nmf_topic_matrix = nmf.fit_transform(X_tfidf)
nmf_labels = nmf_topic_matrix.argmax(axis=1)

In [57]:
nmf_feature_names = tfidf_vectorizer.get_feature_names_out()
print('Ключевые слова (NMF):')
for tid, topic in enumerate(nmf.components_):
    top_words = [nmf_feature_names[i] for i in topic.argsort()[:-9:-1]]
    count = (nmf_labels == tid).sum()
    print(f'Тема {tid} ({count} doc): {", ".join(top_words)}')

Ключевые слова (NMF):
Тема 0 (83 doc): товар, заказ, заказа, деньги, время, итоге, магазина, магазине
Тема 1 (12 doc): самовывоз, платный, рублей, плату, магазин, магазина, деньги, нужный
Тема 2 (32 doc): товара, мало, отсутствие, характеристики, отсутствует, соответствует, нужного, скидки
Тема 3 (71 doc): мтс, магазин, работает, сервис, смартфон, магазине, оператор, просто
Тема 4 (32 doc): наличии, ассортимент, товары, маленький, ничего, очень, товаров, много
Тема 5 (24 doc): доставки, товаров, время, один, бесплатной, могу, ограничен, магазинов
Тема 6 (31 doc): цены, доставка, дорого, платная, высокие, выше, скудный, позиций


### Baseline: KMeans

In [58]:
kmeans = KMeans(n_clusters=n_topics, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(embeddings)

df_km = pd.DataFrame({'text': texts, 'cluster': kmeans_labels})
print('Размеры кластеров (KMeans)')
for cid in sorted(df_km['cluster'].unique()):
    count = (kmeans_labels == cid).sum()
    print(f'Кластер {cid}: {count} doc')

Размеры кластеров (KMeans)
Кластер 0: 25 doc
Кластер 1: 35 doc
Кластер 2: 51 doc
Кластер 3: 35 doc
Кластер 4: 39 doc
Кластер 5: 50 doc
Кластер 6: 50 doc


### Метрики качества

**Silhouette score** [-1, 1]: чем ближе к 1, тем плотнее кластеры и дальше друг от друга. 

**Coherence c_v** [0, 1]: чем выше, тем семантически связнее слова в теме

In [59]:
mask = np.array(topics) != -1
sil_bertopic = silhouette_score(embeddings[mask], np.array(topics)[mask], metric='cosine')
sil_kmeans = silhouette_score(embeddings, kmeans_labels, metric='cosine')

print(f'Silhouette:')
print(f'BERTopic: {sil_bertopic}')
print(f'KMeans: {sil_kmeans}')

Silhouette:
BERTopic: 0.07965654879808426
KMeans: 0.0747118666768074


In [60]:
tokenized = [
    [w for w in text.lower().split()
     if w not in russian_stopwords and len(w) > 2]
    for text in texts
]
dictionary = corpora.Dictionary(tokenized)

def topic_words_bertopic(model, topic_ids, n=10):
    return [[w for w, _ in model.get_topic(tid)][:n] for tid in sorted(topic_ids) if tid != -1]

def topic_words_sklearn(components, feature_names, n=10):
    return [[feature_names[i] for i in t.argsort()[:-n-1:-1]] for t in components]

In [61]:
bt_words = topic_words_bertopic(topic_model, set(topics))
lda_words = topic_words_sklearn(lda.components_, lda_feature_names)
nmf_words = topic_words_sklearn(nmf.components_, nmf_feature_names)

coh_bertopic = CoherenceModel(topics=bt_words,  texts=tokenized, dictionary=dictionary, coherence='c_v').get_coherence()
coh_lda = CoherenceModel(topics=lda_words, texts=tokenized, dictionary=dictionary, coherence='c_v').get_coherence()
coh_nmf = CoherenceModel(topics=nmf_words, texts=tokenized, dictionary=dictionary, coherence='c_v').get_coherence()

print('Coherence:')
print(f'BERTopic: {coh_bertopic}')
print(f'LDA: {coh_lda}')
print(f'NMF: {coh_nmf}')

Coherence:
BERTopic: 0.44544002538111566
LDA: 0.31565017323890104
NMF: 0.40100763267809186


### Наблюдения по сравнению методов

Для Маркетплейса:
- **Лучший Silhouette:** BERTopic (0.0827)
- **Лучший Coherence:** BERTopic (0.457)

Для Видео-платформы:
- **Лучший Silhouette:** KMeans (0.0934)
- **Лучший Coherence:** NMF (0.389)

На сэмпле комментариев из Маркетплейса BERTopic лидирует по обеим метрикам.

На сэмпле комментариев, принадлежащих продукту Видео-платформа, количественные метрики (Silhouette, Coherence) не выявили явного превосходства BERTopic над классическими методами. Абсолютные значения метрик у всех методов невысоки и близки между собой, что говорит об отсутствии явного победителя по формальным критериям. В этих условиях смысловое качество тем становится определяющим критерием выбора.

Качественный анализ показал, что BERTopic даёт более читаемые темы за счёт биграмм и меньшего пересечения слов между кластерами. Кроме того, только BERTopic способен автоматически определять число тем и выделять аномальные документы в шум — что критично при появлении новых, ранее неизвестных проблем. LDA и NMF требуют заранее заданного числа тем и принудительно распределяют все документы, что может маскировать выбросы.

Таким образом, на разных продуктах количественные метрики могут варьироваться, однако при этом качественное преимущество BERTopic остается стабильным.

### Сохранение результатов

In [ ]:
results_dir = Path.cwd().parent / 'results/notebooks'
results_dir.mkdir(exist_ok=True)

product_name = PRODUCT.replace(' ', '_').replace('-', '_').lower()

df_result = df.copy()
df_result['topic'] = topics
df_result.to_csv(results_dir / f'comments_with_topics_{product_name}.csv', index=False, encoding='utf-8', sep=';')

keywords_dict = {}
for _, row in topic_info[topic_info['Topic'] != -1].iterrows():
    tid = int(row['Topic'])
    words = [w for w, _ in topic_model.get_topic(tid)]
    keywords_dict[str(tid)] = words

with open(results_dir / f'topic_keywords_{product_name}.json', 'w', encoding='utf-8') as f:
    json.dump(keywords_dict, f, ensure_ascii=False, indent=2)